# Dynamic Programming Visualization

Interactive visualization of how dynamic programming optimizes recursive algorithms through memoization.

**OCTALUM-PYLAB** - Master Python Data Structures & Algorithms

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import time
from functools import lru_cache

%matplotlib inline

## Part 1: Fibonacci - Naive vs Memoized

The Fibonacci sequence is the classic example to demonstrate DP benefits.

- **Naive recursion**: O(2^n) - exponential!
- **Memoization**: O(n) - linear!

In [ ]:
# Track function calls for visualization
call_counts = {'naive': 0, 'memoized': 0}

def fib_naive(n: int) -> int:
    """Naive recursive Fibonacci - exponential time complexity."""
    call_counts['naive'] += 1
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

def fib_memoized(n: int, memo: dict[int, int] | None = None) -> int:
    """Memoized Fibonacci - linear time complexity."""
    if memo is None:
        memo = {0: 0, 1: 1}
        call_counts['memoized'] = 0
    
    call_counts['memoized'] += 1
    
    if n in memo:
        return memo[n]
    
    memo[n] = fib_memoized(n - 1, memo) + fib_memoized(n - 2, memo)
    return memo[n]

### Compare Call Counts

In [ ]:
# Test both approaches and collect data
test_values = list(range(5, 26, 5))
naive_calls = []
memoized_calls = []
naive_times = []
memoized_times = []

for n in test_values:
    # Naive
    call_counts['naive'] = 0
    start = time.perf_counter()
    fib_naive(n)
    naive_calls.append(call_counts['naive'])
    naive_times.append(time.perf_counter() - start)
    
    # Memoized
    call_counts['memoized'] = 0
    start = time.perf_counter()
    fib_memoized(n)
    memoized_calls.append(call_counts['memoized'])
    memoized_times.append(time.perf_counter() - start)

print(f"{'n':<5} {'Naive Calls':<15} {'Memoized Calls':<15} {'Speedup':<10}")
print("-" * 50)
for i, n in enumerate(test_values):
    speedup = naive_calls[i] / memoized_calls[i] if memoized_calls[i] > 0 else 0
    print(f"{n:<5} {naive_calls[i]:<15} {memoized_calls[i]:<15} {speedup:<10.1f}x")

In [ ]:
# Visualize call counts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Call count comparison
x = np.arange(len(test_values))
width = 0.35

ax1.bar(x - width/2, naive_calls, width, label='Naive O(2^n)', color='#ff6b6b')
ax1.bar(x + width/2, memoized_calls, width, label='Memoized O(n)', color='#4ecdc4')
ax1.set_xlabel('n')
ax1.set_ylabel('Function Calls')
ax1.set_title('Fibonacci: Function Call Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(test_values)
ax1.legend()
ax1.set_yscale('log')

# Execution time comparison
ax2.plot(test_values, naive_times, 'o-', label='Naive', color='#ff6b6b', linewidth=2)
ax2.plot(test_values, memoized_times, 'o-', label='Memoized', color='#4ecdc4', linewidth=2)
ax2.set_xlabel('n')
ax2.set_ylabel('Time (seconds)')
ax2.set_title('Fibonacci: Execution Time')
ax2.legend()
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

## Part 2: 0/1 Knapsack Problem

Given items with weights and values, maximize value while staying within weight capacity.

- **DP Table**: rows = items, columns = capacity
- **Transition**: `dp[i][w] = max(dp[i-1][w], dp[i-1][w-weight[i]] + value[i])`

In [ ]:
def knapsack_dp(weights: list[int], values: list[int], capacity: int) -> tuple[int, list[list[int]]]:
    """Solve 0/1 Knapsack and return max value + DP table for visualization."""
    n = len(weights)
    # DP table: dp[i][w] = max value using first i items with capacity w
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]
    
    for i in range(1, n + 1):
        for w in range(capacity + 1):
            # Don't take item i
            dp[i][w] = dp[i - 1][w]
            # Take item i (if it fits)
            if weights[i - 1] <= w:
                dp[i][w] = max(dp[i][w], dp[i - 1][w - weights[i - 1]] + values[i - 1])
    
    return dp[n][capacity], dp

In [ ]:
# Example: 4 items
weights = [2, 3, 4, 5]
values = [3, 4, 5, 6]
capacity = 8
item_names = ['Item A', 'Item B', 'Item C', 'Item D']

max_value, dp_table = knapsack_dp(weights, values, capacity)

print("Items:")
for i, (name, w, v) in enumerate(zip(item_names, weights, values)):
    print(f"  {name}: weight={w}, value={v}")
print(f"\nCapacity: {capacity}")
print(f"Maximum value: {max_value}")

In [ ]:
# Visualize DP table
fig, ax = plt.subplots(figsize=(12, 6))

dp_array = np.array(dp_table)
im = ax.imshow(dp_array, cmap='YlOrRd', aspect='auto')

# Labels
ax.set_xlabel('Capacity')
ax.set_ylabel('Items Considered')
ax.set_title('Knapsack DP Table Visualization')
ax.set_xticks(range(capacity + 1))
ax.set_yticks(range(len(weights) + 1))
ax.set_yticklabels(['None'] + item_names)

# Add value annotations
for i in range(len(weights) + 1):
    for j in range(capacity + 1):
        text_color = 'white' if dp_array[i, j] > dp_array.max() / 2 else 'black'
        ax.text(j, i, dp_array[i, j], ha='center', va='center', color=text_color)

plt.colorbar(im, label='Max Value')
plt.tight_layout()
plt.show()

## Part 3: Complexity Visualization

Compare exponential O(2^n) vs polynomial O(n^2) vs linear O(n) growth.

In [ ]:
n_values = np.arange(1, 21)

exponential = 2 ** n_values
quadratic = n_values ** 2
linear = n_values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax1.plot(n_values, exponential, 'o-', label='O(2^n) - Exponential', color='#e74c3c', linewidth=2)
ax1.plot(n_values, quadratic, 's-', label='O(n^2) - Quadratic', color='#f39c12', linewidth=2)
ax1.plot(n_values, linear, '^-', label='O(n) - Linear', color='#27ae60', linewidth=2)
ax1.set_xlabel('Input Size (n)')
ax1.set_ylabel('Operations')
ax1.set_title('Algorithm Complexity Comparison (Linear Scale)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Log scale
ax2.semilogy(n_values, exponential, 'o-', label='O(2^n) - Exponential', color='#e74c3c', linewidth=2)
ax2.semilogy(n_values, quadratic, 's-', label='O(n^2) - Quadratic', color='#f39c12', linewidth=2)
ax2.semilogy(n_values, linear, '^-', label='O(n) - Linear', color='#27ae60', linewidth=2)
ax2.set_xlabel('Input Size (n)')
ax2.set_ylabel('Operations (log scale)')
ax2.set_title('Algorithm Complexity Comparison (Log Scale)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Memoization** transforms exponential algorithms into polynomial/linear by caching results
2. **DP Table** visualizes how subproblems build up to the final solution
3. **Overlapping Subproblems** - DP shines when the same subproblems are solved multiple times
4. **Optimal Substructure** - optimal solution contains optimal solutions to subproblems

### When to Use DP:
- Counting problems (number of ways)
- Optimization problems (min/max)
- Decision problems (yes/no)
- Problems with overlapping subproblems